In [ ]:
!pip install torch-geometric

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import time
import copy
import random
import pandas as pd

from torch_geometric.datasets import LRGBDataset
from torch_geometric.loader import DataLoader
from torch_geometric.utils import to_dense_adj

from sklearn.metrics import average_precision_score
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)


In [ ]:
lrg_train = LRGBDataset(root="data/LRGB", name="Peptides-func", split="train")
lrg_val   = LRGBDataset(root="data/LRGB", name="Peptides-func", split="val")
lrg_test  = LRGBDataset(root="data/LRGB", name="Peptides-func", split="test")

lrg_train_loader = DataLoader(lrg_train, batch_size=1, shuffle=True)
lrg_val_loader   = DataLoader(lrg_val, batch_size=1)
lrg_test_loader  = DataLoader(lrg_test, batch_size=1)


In [ ]:
def compute_U(A):
    L = torch.diag(A.sum(dim=1)) - A
    eigvals, eigvecs = torch.linalg.eigh(L)
    return eigvecs


In [ ]:
class DenseGCNLayer(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.linear = nn.Linear(dim, dim)

    def forward(self, x, A_norm):
        return self.linear(A_norm @ x)


In [ ]:
class SpectralMix(nn.Module):
    def __init__(self, dim, k=None, mode="low"):
        super().__init__()
        self.k = k
        self.mode = mode

        self.mlp = nn.Sequential(
            nn.Linear(dim, dim),
            nn.GELU(),
            nn.Linear(dim, dim)
        )
        self.norm = nn.LayerNorm(dim)

    def forward(self, x, U, mask=None):

        if self.k is not None:
            if self.mode == "low":
                U = U[:, :, :self.k]
            else:
                U = U[:, :, -self.k:]

        x_hat = torch.matmul(U.transpose(1,2), x)
        x_hat = self.mlp(x_hat)
        x_out = torch.matmul(U, x_hat)

        out = self.norm(x + x_out)

        if mask is not None:
            out = out * mask.unsqueeze(-1)

        return out


In [ ]:
class HybridGraphFNet(nn.Module):
    def __init__(self, in_dim, hidden_dim=64, num_layers=3, out_dim=10, k=None):
        super().__init__()
        self.input_proj = nn.Linear(in_dim, hidden_dim)
        self.layers = nn.ModuleList([
            nn.ModuleDict({
                "local": DenseGCNLayer(hidden_dim),
                "global": SpectralMix(hidden_dim, k=k)
            })
            for _ in range(num_layers)
        ])
        self.classifier = nn.Linear(hidden_dim, out_dim)

    def forward(self, data):

    # -------- 1. Convert to dense batch --------
        x, mask = to_dense_batch(data.x.float(), data.batch)   # [B, N, D]
        adj = to_dense_adj(data.edge_index, data.batch, max_num_nodes=x.size(1))  # [B, N, N]

        # -------- 2. Add self loops --------
        I = torch.eye(adj.size(1), device=x.device).unsqueeze(0)
        adj = adj + I

        # -------- 3. Compute normalized adjacency + eigenbasis --------
        A_norm, U = self.compute_laplacian_basis(adj, mask)

        # -------- 4. Input projection --------
        x = self.input_proj(x)

        # -------- 5. Apply layers --------
        for idx, layer in enumerate(self.layers):
            x_local = layer["local"](x, A_norm)
            x_global = layer["global"](x, U, mask)

            alpha = torch.sigmoid(self.alphas[idx])
            x = alpha * x_local + (1 - alpha) * x_global

        # -------- 6. Mask padded nodes --------
        x = x * mask.unsqueeze(-1)

        # -------- 7. Graph pooling --------
        sum_pooled = x.sum(dim=1)
        num_nodes = mask.sum(dim=1, keepdim=True)
        graph_emb = sum_pooled / (num_nodes + 1e-6)

        return self.classifier(graph_emb)


In [ ]:
def evaluate(model, loader, task):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            out = model(data)
            if task == "regression":
                all_preds.append(out.cpu())
                all_labels.append(data.y.view(-1).cpu())
            else:
                all_preds.append(out.cpu())
                all_labels.append(data.y.float().view(-1).cpu())
    preds = torch.cat(all_preds)
    labels = torch.cat(all_labels)
    if task == "regression":
        return (preds - labels).abs().mean().item()
    else:
        return average_precision_score(labels.numpy(), preds.numpy())


In [ ]:
def train_model(model, train_loader, val_loader,
                task="classification",
                max_epochs=100, patience=10):

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    best_val = float("inf") if task=="regression" else -1
    best_state = None
    patience_counter = 0
    best_epoch = 0
    start_time = time.time()
    print(f"\nStarting training | Task: {task} | Max Epochs: {max_epochs}")

    for epoch in range(1, max_epochs + 1):
        model.train()
        total_loss = 0
        for data in train_loader:
            data = data.to(device)
            optimizer.zero_grad()
            out = model(data)
            if task == "regression":
                loss = F.l1_loss(out, data.y.view(-1))
            else:
                loss = F.binary_cross_entropy_with_logits(
                    out.view(-1), data.y.float().view(-1)
                )
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        val_metric = evaluate(model, val_loader, task)
        improved = (
            val_metric < best_val if task=="regression"
            else val_metric > best_val
        )
        if improved:
            best_val = val_metric
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
            best_epoch = epoch
        else:
            patience_counter += 1

        print(f"Epoch {epoch} | Loss: {avg_loss:.4f} | Val: {val_metric:.4f}")

        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

    end_time = time.time()
    training_time = end_time - start_time
    model.load_state_dict(best_state)
    return model, best_val, best_epoch, training_time


In [ ]:
def measure_model_memory(model, sample_batch, loss_fn, optimizer):
    model.train()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    inputs = sample_batch
    inputs = inputs.to(device)
    outputs = model(inputs)
    loss = loss_fn(outputs.view(-1), inputs.y.float().view(-1))
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    torch.cuda.synchronize()
    peak = torch.cuda.max_memory_allocated() / (1024 ** 3)
    return peak


In [ ]:
# ── Run k=32, seed=0 ──
SEED = 0
K    = 32

set_seed(SEED)

model = HybridGraphFNet(
    in_dim=lrg_train.num_features,
    out_dim=lrg_train.num_classes,
    k=K
).to(device)

# Measure peak GPU memory before full training
sample_batch = next(iter(lrg_train_loader))
optimizer_mem = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = torch.nn.BCEWithLogitsLoss()
peak_mem = measure_model_memory(model, sample_batch, loss_fn, optimizer_mem)
print(f"[MODEL MEMORY][k={K}][seed={SEED}] Peak GPU: {peak_mem:.4f} GB")

# Re-init model cleanly after memory probe
set_seed(SEED)
model = HybridGraphFNet(
    in_dim=lrg_train.num_features,
    out_dim=lrg_train.num_classes,
    k=K
).to(device)

model, val_ap, best_epoch, train_time = train_model(
    model, lrg_train_loader, lrg_val_loader, task="classification"
)

test_ap = evaluate(model, lrg_test_loader, "classification")

save_path = f"model_k{K}_seed{SEED}.pt"
torch.save(model.state_dict(), save_path)
print(f"Model saved: {save_path}")

results_k32_seed0 = {
    "seed": SEED,
    "k": K,
    "val_ap": val_ap,
    "test_ap": test_ap,
    "best_epoch": best_epoch,
    "train_time": train_time,
    "memory_gb": peak_mem
}

print(f"""
========== FINAL RESULTS ==========
Seed       : {SEED}
k          : {K}
Val AP     : {val_ap:.4f}
Test AP    : {test_ap:.4f}
Best Epoch : {best_epoch}
Train Time : {train_time:.2f}s
Peak Mem   : {peak_mem:.4f} GB
===================================
""")


In [ ]:
# Save results to CSV
df = pd.DataFrame([results_k32_seed0])
csv_path = f"results_k32_seed0.csv"
df.to_csv(csv_path, index=False)
print(f"Results saved to {csv_path}")
display(df)

# Build lrg_results dict compatible with downstream cells
lrg_results = {
    32: {
        "mean": results_k32_seed0["test_ap"],
        "std":  0.0,
        "logs": [results_k32_seed0]
    }
}


In [ ]:
# Fixed: access 'mean' key instead of index [0]
best_k = max(lrg_results, key=lambda x: lrg_results[x]["mean"])
print("Best k:", best_k)
print(f"Test AP (mean): {lrg_results[best_k]['mean']:.4f}")


In [ ]:
rows = []
for k, v in lrg_results.items():
    rows.append({"k": k, "mean_test_ap": v["mean"], "std_test_ap": v["std"]})
summary_df = pd.DataFrame(rows)
summary_df.to_csv("lrg_k_sweep_results.csv", index=False)
print("Saved lrg_k_sweep_results.csv")
display(summary_df)
